# PhoBERT Hybrid Explainable v3

Notebook Kaggle chạy branch `feature/improve-training-v3-explainable`: split ACD/SPC, threshold tuning, rare oversampling, top-k checkpoint và ensemble.

In [ ]:
import os, sys, subprocess, shutil, json

REPO_URL = 'https://github.com/vudinhminh08/NLP-project-master-study.git'
BRANCH = 'feature/improve-training-v3-explainable'
PROJECT_DIR = '/kaggle/working/absa-project'

if os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    subprocess.check_call(['git', '-C', PROJECT_DIR, 'fetch', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', PROJECT_DIR, 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', PROJECT_DIR, 'pull', '--ff-only', 'origin', BRANCH])
else:
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, PROJECT_DIR])

os.chdir(PROJECT_DIR)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

sys.path.insert(0, os.path.join(PROJECT_DIR, 'code', 'phobert'))
sys.path.insert(0, os.path.join(PROJECT_DIR, 'code', 'data_processing'))
print('Working dir:', os.getcwd())
print('Branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    torch.cuda.empty_cache()

In [ ]:
from run_experiment import main

V3_BASE = {
    'use_split_loss': True,
    'use_attention_pooling': False,
    'tune_presence_threshold': True,
    'use_rare_oversampling': True,
}

v3_cls_metrics = main(
    encoder_option='cls_only',
    use_amp=True,
    lr=1e-4,
    max_epochs=20,
    early_stop_patience=7,
    run_name='v3_cls_split_lr1e4',
    config_overrides=V3_BASE,
)
print('v3_cls_split_lr1e4 primary Combined F1:', v3_cls_metrics['primary_combined_f1'])

In [ ]:
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

v3_concat_metrics = main(
    encoder_option='concat_4_layers',
    use_amp=True,
    lr=1e-4,
    max_epochs=20,
    early_stop_patience=7,
    run_name='v3_concat_split_lr1e4',
    config_overrides=V3_BASE,
)
print('v3_concat_split_lr1e4 primary Combined F1:', v3_concat_metrics['primary_combined_f1'])

In [ ]:
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

v3_cls_lr2e5_metrics = main(
    encoder_option='cls_only',
    use_amp=True,
    lr=2e-5,
    max_epochs=40,
    early_stop_patience=10,
    run_name='v3_cls_split_lr2e5',
    config_overrides=V3_BASE,
)
print('v3_cls_split_lr2e5 primary Combined F1:', v3_cls_lr2e5_metrics['primary_combined_f1'])

In [ ]:
# Optional ablation: attention pooling. Run only if you still have GPU time.
RUN_ATTENTION_ABLATION = False
if RUN_ATTENTION_ABLATION:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    v3_attention_metrics = main(
        encoder_option='cls_only',
        use_amp=True,
        lr=1e-4,
        max_epochs=20,
        early_stop_patience=7,
        run_name='v3_cls_attention_lr1e4',
        config_overrides={**V3_BASE, 'use_attention_pooling': True},
    )
    print('v3_cls_attention_lr1e4 primary Combined F1:', v3_attention_metrics['primary_combined_f1'])

In [ ]:
import json, os
import matplotlib.pyplot as plt

RUNS = {
    'v3 cls split lr=1e-4': 'outputs/results_v3_cls_split_lr1e4/training_history.json',
    'v3 concat split lr=1e-4': 'outputs/results_v3_concat_split_lr1e4/training_history.json',
    'v3 cls split lr=2e-5': 'outputs/results_v3_cls_split_lr2e5/training_history.json',
    'v3 cls attention lr=1e-4': 'outputs/results_v3_cls_attention_lr1e4/training_history.json',
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for label, path in RUNS.items():
    if not os.path.exists(path):
        continue
    h = json.load(open(path, encoding='utf-8'))
    epochs = range(1, len(h['train_loss']) + 1)
    axes[0].plot(epochs, h['train_loss'], marker='o', label=f'{label} train')
    axes[0].plot(epochs, h['dev_loss'], marker='s', label=f'{label} dev')
    axes[1].plot(epochs, h['dev_combined_f1'], marker='o', label=label)
    axes[1].axvline(h['best_epoch'], linestyle='--', alpha=0.25)

axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[1].set_title('Dev Combined F1')
axes[1].set_xlabel('Epoch')
axes[1].axhline(0.5543, color='gray', linestyle=':', label='Baseline 0.5543')
axes[1].axhline(0.6218, color='purple', linestyle=':', label='Reference best 0.6218')
axes[1].axhline(0.7732, color='red', linestyle=':', label='SOTA 0.7732')
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
os.makedirs('outputs/eda', exist_ok=True)
plt.savefig('outputs/eda/learning_curve_v3.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import os, json
import pandas as pd

REFERENCE_ROWS = [
    {'Run': 'Baseline old cls_only lr=1e-4', 'ACD F1': 0.6360, 'SPC F1': 0.4727, 'Combined F1': 0.5543},
    {'Run': 'Reference concat', 'ACD F1': 0.6827, 'SPC F1': 0.5374, 'Combined F1': 0.6101},
    {'Run': 'Reference cls_only', 'ACD F1': 0.6849, 'SPC F1': 0.5587, 'Combined F1': 0.6218},
]

def add_metrics(rows, name, path):
    if not os.path.exists(path):
        return
    m = json.load(open(path, encoding='utf-8'))
    rows.append({
        'Run': name,
        'ACD F1': m['macro_acd_f1'],
        'SPC F1': m['macro_spc_f1'],
        'Combined F1': m['macro_combined_f1'],
    })

rows = list(REFERENCE_ROWS)
for run in ['v3_cls_split_lr1e4', 'v3_concat_split_lr1e4', 'v3_cls_split_lr2e5', 'v3_cls_attention_lr1e4']:
    add_metrics(rows, f'{run} single', f'outputs/results_{run}/phobert_test_metrics.json')
    add_metrics(rows, f'{run} ensemble', f'outputs/results_{run}/phobert_test_ensemble_metrics.json')
rows.append({'Run': 'SOTA ds4v 2022', 'ACD F1': 0.8255, 'SPC F1': None, 'Combined F1': 0.7732})

df = pd.DataFrame(rows)
display(df)
candidates = df[df['Run'].str.startswith('v3_')].dropna(subset=['Combined F1'])
if len(candidates):
    primary = candidates.sort_values('Combined F1', ascending=False).iloc[0]
    print('PRIMARY_RESULT =', primary['Run'])
    print('PRIMARY_COMBINED_F1 =', f"{primary['Combined F1']:.4f}")
df.to_csv('outputs/results_v3_comparison.csv', index=False)

In [ ]:
import shutil
zip_path = shutil.make_archive('/kaggle/working/phobert_v3_results', 'zip', 'outputs')
print('Zip:', zip_path)